<a href="https://colab.research.google.com/github/Tamur-Naseem/FLY-RANK/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tamur-Naseem/FLY-RANK/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*
**Finding 1:** "Models trained on content staleness achieved a high accuracy in predicting traffic decline."
*   **Methodology Question (Validation Design):** Does the validation split group by client? If a standard random split was used and a specific client suffered a site-wide algorithmic penalty, that client's pages would end up in both the training and test sets. The model might just be memorizing that specific client's baseline rather than learning a universal rule about staleness.

**Finding 2:** "Pages flagged with a low health score are 3x more likely to lose traffic."
*   **Methodology Question (Label Origin):** Exactly what inputs define the "health score"? If this score is calculated by the product using overlapping time windows or future data, using it as a feature to predict future traffic loss introduces target leakage. We need to verify it is built entirely on historical, observable metrics.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# No code required for this section. The analysis is purely methodological.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*
**The Honest Split:** I am switching from a random `train_test_split` to a `GroupShuffleSplit` grouped by `client_id`.
A random split allows pages from the same client to exist in both training and testing. Grouping by client guarantees that all pages for a specific client are entirely in the test set, proving whether the model can generalize its logic to a brand new, unseen client.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

# Load data directly from GitHub
url = 'https://raw.githubusercontent.com/Tamur-Naseem/FLY-RANK/main/data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(url)

# Setup Target and Features
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)
features = ['content_age_days', 'impressions_90d', 'sessions_90d', 'avg_position', 'ctr', 'word_count']
X = df[features].fillna(0)
y = df['is_declining']
groups = df['client_id']

# --- BEFORE: Random Split (Week 5) ---
X_train_rnd, X_test_rnd, y_train_rnd, y_test_rnd = train_test_split(X, y, test_size=0.2, random_state=42)
rf_random = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
rf_random.fit(X_train_rnd, y_train_rnd)

test_rnd = X_test_rnd.copy()
test_rnd['y_true'] = y_test_rnd
test_rnd['prob'] = rf_random.predict_proba(X_test_rnd)[:, 1]
p50_random = test_rnd.sort_values('prob', ascending=False).head(50)['y_true'].mean()

# --- AFTER: Grouped Split (Honest) ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train_grp, X_test_grp = X.iloc[train_idx], X.iloc[test_idx]
y_train_grp, y_test_grp = y.iloc[train_idx], y.iloc[test_idx]

rf_grouped = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
rf_grouped.fit(X_train_grp, y_train_grp)

test_grp = X_test_grp.copy()
test_grp['y_true'] = y_test_grp
test_grp['prob'] = rf_grouped.predict_proba(X_test_grp)[:, 1]
p50_grouped = test_grp.sort_values('prob', ascending=False).head(50)['y_true'].mean()

print("--- SPLIT COMPARISON (Precision@50) ---")
print(f"Random Split (Unseen Rows, seen clients):    {p50_random:.3f}")
print(f"Grouped Split (Unseen Rows, UNSEEN clients): {p50_grouped:.3f}")
print("\nConclusion: The metric usually drops under a grouped split, exposing how much the model 'cheated' by memorizing client patterns. This new, lower number is our honest baseline.")


--- SPLIT COMPARISON (Precision@50) ---
Random Split (Unseen Rows, seen clients):    0.880
Grouped Split (Unseen Rows, UNSEEN clients): 0.580

Conclusion: The metric usually drops under a grouped split, exposing how much the model 'cheated' by memorizing client patterns. This new, lower number is our honest baseline.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*
**Audit Strategy:** First, I programmatically verify that no derived product flags or target-leaking variables (like the `trend_direction` itself) made it into my feature array. Second, I isolate the top 5 false positives (pages the model was highly confident would decline, but didn't) to see what context the model is missing.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 1. Feature Leakage Audit
forbidden_terms = ['trend', 'score', 'flag', 'future', 'declining']
leaked_features = [col for col in X.columns if any(term in col.lower() for term in forbidden_terms)]

print("--- LEAKAGE AUDIT ---")
if not leaked_features:
    print("PASS: No forbidden product flags or derived target variables found in features.")
else:
    print(f"FAIL: Found potentially leaking features: {leaked_features}")

# 2. Error Analysis (Top 5 False Positives from the Grouped Split)
false_positives = test_grp[(test_grp['prob'] > 0.6) & (test_grp['y_true'] == 0)].sort_values('prob', ascending=False).head(5)

print("\n--- FAILURE EXAMPLES (False Positives) ---")
if not false_positives.empty:
    print("These pages scored high for decline risk, but remained stable:")
    for i, row in enumerate(false_positives.itertuples(), 1):
        print(f"  {i}. Prob: {row.prob:.2f} | Age: {row.content_age_days:.0f}d | Impr: {row.impressions_90d:.0f} | Pos: {row.avg_position:.1f}")
    print("\nInterpretation: The model likely flags high-impression, older pages as declining when they are actually just highly seasonal. Adding a 'seasonality' metric could resolve this blind spot.")
else:
    print("No high-confidence false positives found in this slice.")


--- LEAKAGE AUDIT ---
PASS: No forbidden product flags or derived target variables found in features.

--- FAILURE EXAMPLES (False Positives) ---
These pages scored high for decline risk, but remained stable:
  1. Prob: 0.79 | Age: 174d | Impr: 101 | Pos: 23.1
  2. Prob: 0.77 | Age: 275d | Impr: 3369 | Pos: 13.2
  3. Prob: 0.76 | Age: 271d | Impr: 264 | Pos: 22.2
  4. Prob: 0.76 | Age: 271d | Impr: 330 | Pos: 25.0
  5. Prob: 0.76 | Age: 238d | Impr: 1191 | Pos: 23.1

Interpretation: The model likely flags high-impression, older pages as declining when they are actually just highly seasonal. Adding a 'seasonality' metric could resolve this blind spot.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*
**Unsafe/Overblown Claim:**
"Our machine learning model proves that rewriting pages older than 180 days will completely stop traffic declines and recover lost users."

**Honest, Public-Safe Rewrite:**
"We observed that historical content age and impression volume act as directional signals for potential traffic decay. This model serves as a decision-support tool to rank refresh candidates for human review; it does not guarantee causal traffic recovery."

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# No code required for this section.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.